<a href="https://colab.research.google.com/github/Karna14314/Hugginface_models/blob/main/PersonaChat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install transformers datasets accelerate peft

Now that the libraries are installed, let's load the PersonaChat dataset and prepare it for fine-tuning DialoGPT.

In [ ]:
from datasets import load_dataset

# Loading an alternative accessible dialogue dataset mirror
try:
    dataset = load_dataset("gandis/persona_chat_augmented", split="train[:1000]")
except Exception as e:
    print(f"Primary fallback failed: {e}. Trying secondary fallback...")
    # Using a widely available generic dialogue dataset
    dataset = load_dataset("fka/awesome-chatgpt-prompts", split="train")

display(dataset)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # For causal LM training, labels should be same as input_ids
    outputs = tokenizer(examples['prompt'], padding='max_length', truncation=True, max_length=128)
    outputs['labels'] = outputs['input_ids'].copy()
    return outputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("Dataset tokenized successfully!")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import torch
import os

# Load base model
model = AutoModelForCausalLM.from_pretrained(model_name)

# Configure LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Define training arguments
training_args = TrainingArguments(
    output_dir="./dialogpt-tiny-chatbot",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=100,
    save_total_limit=1,
    logging_steps=10,
    learning_rate=5e-4,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    report_to="none",
    remove_unused_columns=True
)

# Use a data collator to handle padding for us
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Filter tokenized dataset to only include columns used by the model
# This prevents the 'act' column and other strings from causing tensor errors
columns_to_keep = ["input_ids", "attention_mask", "labels"]
final_train_dataset = tokenized_dataset.remove_columns([col for col in tokenized_dataset.column_names if col not in columns_to_keep])

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_train_dataset,
    data_collator=data_collator,
)

print("Starting training with cleaned dataset...")
trainer.train()

In [ ]:
# Save the fine-tuned model
import os
if 'trainer' in globals():
    save_path = "final_chatbot_model"
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Model saved successfully to {os.path.abspath(save_path)}!")
else:
    print("Error: 'trainer' not found. Please ensure the training cell (4f37fc5a) finished successfully.")

In [ ]:
from transformers import pipeline
import os
import torch

# Use absolute path to avoid 'Repo id' validation errors for local folders
model_path = os.path.abspath("final_chatbot_model")

if os.path.exists(model_path):
    print(f"Loading model from: {model_path}")
    chatbot = pipeline("text-generation", model=model_path, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

    def chat_with_bot(user_input):
        # Use a higher max_new_tokens for better responses
        response = chatbot(user_input, max_new_tokens=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)
        return response[0]['generated_text']

    # Test input
    print(f"Bot: {chat_with_bot('Hello! Who are you?')}")
else:
    print("Model directory not found. Please run the saving cell (aebe44e1) first.")

## Step 6: Upload to Hugging Face
To share your model, you need to log in to Hugging Face. Use a token with 'Write' permissions.

In [ ]:
from huggingface_hub import notebook_login

# This will prompt you for your Hugging Face token
notebook_login()

In [ ]:
# Replace 'your-username' with your actual Hugging Face username
hf_username = "ncncomplete"
repo_id = "ncncomplete/dialogpt-tiny-chatbot"

# Push the model and tokenizer
if hf_username != "your-username":
    print(f"Uploading to {repo_id}...")
    trainer.model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)
    print(f"Model successfully uploaded to: https://huggingface.co/{repo_id}")
else:
    print("Error: Please update the 'hf_username' variable with your actual Hugging Face username.")